# Computational Theory: SHA-256

This notebook builds SHA-256 in Python with NumPy, following the [Secure Hash Standard (FIPS 180-4)](https://nvlpubs.nist.gov/nistpubs/FIPS/NIST.FIPS.180-4.pdf).

## Problem 1: Represent SHA-256 data in Python

### 32-bit word

A 32-bit word is a group of 32 binary digits (bits), treated as one unit by the algorithm. For example, `0x00000001` (represents decimal 1) and `0xFFFFFFFF` (represents decimal $2^{32}-1$) are both 32-bit words: each has eight hexadecimal digits, with each digit representing four bits.

The standard defines SHA-256 in terms of 32-bit words ([FIPS 180-4, Section 3.1, p.7](https://nvlpubs.nist.gov/nistpubs/FIPS/NIST.FIPS.180-4.pdf#page=12)).
A word contains exactly 32 bits, so its unsigned integer value is in the following range:

$$
0 \leq w \leq 2^{32} - 1.
$$

NumPy's `uint32` type represents values in this range directly ([NumPy data types](https://numpy.org/doc/stable/user/basics.types.html)). This fixed-width representation is useful because SHA-256 arithmetic is performed on 32-bit words; in particular, addition is modulo $2^{32}$ ([FIPS 180-4, Section 4.1.2, p.10](https://nvlpubs.nist.gov/nistpubs/FIPS/NIST.FIPS.180-4.pdf#page=15)). Python's built-in `int` has arbitrary precision, so using `uint32` makes the intended word width explicit.

In [1]:
import numpy as np

# Start with the largest value that fits in an unsigned 32-bit word.
word = np.uint32(0xFFFFFFFF)
# Represent 1 as a 32-bit word too, so both operands have the same width.
one = np.uint32(1)

# Force the result to remain uint32: overflow wraps modulo 2**32.
# ** means exponentiation in Python, so 2**32 is 2 raised to the power of 32.
# See: https://numpy.org/doc/stable/reference/generated/numpy.add.html
wrapped = np.add(word, one, dtype=np.uint32)

# Convert to Python ints for formatting as eight-digit hexadecimal values.
print(f"word    = 0x{int(word):08X}")
print(f"word + 1 = 0x{int(wrapped):08X}")
# Check that the maximum 32-bit value plus 1 wraps around to zero.
assert wrapped == np.uint32(0)

word    = 0xFFFFFFFF
word + 1 = 0x00000000


### Sequences of 32-bit words

#### What is a sequence?

Think of a word as one box. The box holds one number, stored as 32 on/off switches (bits). A sequence of words is just a row of those boxes. The boxes are numbered from $0$:

```text
position:     0          1          2          3
          +----------+----------+----------+----------+
          |   word   |   word   |   word   |   word   |
          +----------+----------+----------+----------+
```

SHA-256 does not hash one box at a time. It takes rows of boxes, mixes them together, and hands back a row of 8 boxes: the hash. So we need a way to store "a row of 32-bit boxes" in Python.

#### How to store a row of boxes in Python

Use NumPy arrays ([`numpy.array`](https://numpy.org/doc/stable/reference/generated/numpy.array.html)) and always write `dtype=np.uint32` ([NumPy `dtype` reference](https://numpy.org/doc/stable/reference/arrays.dtypes.html)). Without it, NumPy picks bigger 64-bit boxes, and SHA-256 needs exactly 32. Example: `np.array([...], dtype=np.uint32)` means:

| Term | Plain meaning |
|:---|:---|
| `dtype` | "data type": what kind of thing every box holds. |
| `np.uint32` | The kind of box we want: a whole number made of 32 switches that can't be negative. u = unsigned, int = whole number, 32 = number of bits.|

In [2]:
import numpy as np

# A row of 4 boxes, each a 32-bit whole number.
# See: https://numpy.org/doc/stable/reference/generated/numpy.array.html
row = np.array([10, 20, 30, 40], dtype=np.uint32)

print(row)        # all the boxes
print(row[0])     # the first box (position 0)
print(row[3])     # the last box (position 3)
print(row.size)   # how many boxes
print(row.shape)  # (4,) means: one row with 4 boxes
print(row.dtype)  # what each box holds

[10 20 30 40]
10
40
4
(4,)
uint32


#### The four rows SHA-256 uses

The standard defines these four rows: the message block $M$ ([Section 5.2.1, p.14](https://nvlpubs.nist.gov/nistpubs/FIPS/NIST.FIPS.180-4.pdf#page=19)), the round constants $K$ ([Section 4.2.2, p.11](https://nvlpubs.nist.gov/nistpubs/FIPS/NIST.FIPS.180-4.pdf#page=16)), and the message schedule $W$ and hash value $H$ ([Section 6.2, p.21](https://nvlpubs.nist.gov/nistpubs/FIPS/NIST.FIPS.180-4.pdf#page=26)). Every box is 32 bits, so the size in bits is just the number of boxes $\times\ 32$.

| Row | What it is for | Boxes | Bits | Math name |
|:---|:---|:---:|:---:|:---:|
| Message block | Your message, cut into pieces. SHA-256 reads 512 bits at a time, so a long message is split into 512-bit pieces and each piece fills one row of 16 boxes. | 16 | 512 | $M_0, \dots, M_{15}$ |
| Message schedule | For each piece, SHA-256 stretches its 16 boxes into 64 boxes by mixing the earlier boxes together. The algorithm has 64 steps, and each step uses one of these boxes. | 64 | 2048 | $W_0, \dots, W_{63}$ |
| Hash value | The answer so far. It starts as 8 fixed numbers, is updated after every piece, and once the last piece is done these 8 boxes are the final hash ($8 \times 32 = 256$ bits). | 8 | 256 | $H_0, \dots, H_7$ |
| Round constants | A fixed list of 64 numbers, one for each of the 64 steps, copied from the standard. They are the same for every message and are mixed in at each step to scramble the numbers. | 64 | 2048 | $K_0, \dots, K_{63}$ |

Reading the math names: the letter names the row, the small number below it is the box position. So $W_5$ means "box 5 of the message schedule" and $H_0, \dots, H_7$ means "boxes 0, 1, 2, and so on up to 7 of the hash value". The standard also writes a small $(i)$ above the letter to say which block or hash value.

Hexadecimal: the standard writes each box as 8 hex digits, because each digit is 4 bits and $8 \times 4 = 32$.

The code below builds two of the rows:

- the message block for the text `"abc"` ([Section 5.1.1, p.13](https://nvlpubs.nist.gov/nistpubs/FIPS/NIST.FIPS.180-4.pdf#page=18); the full worked example is in the [NIST SHA-256 example](https://csrc.nist.gov/CSRC/media/Projects/Cryptographic-Standards-and-Guidelines/documents/examples/SHA256.pdf))
- the starting hash value ([Section 5.3.3, p.15](https://nvlpubs.nist.gov/nistpubs/FIPS/NIST.FIPS.180-4.pdf#page=20))

In [3]:
import numpy as np

# Message block: 16 empty boxes (all zeros) to start with.
block = np.zeros(16, dtype=np.uint32)
# Box 0 holds the letters a, b, c (61 62 63) and then a single 1 bit (the 80).
block[0] = 0x61626380
# Box 15 holds the message length: 3 letters x 8 bits = 24 bits = 0x18.
block[15] = 0x00000018

# Starting hash value: 8 boxes, given in the standard.
H0 = np.array([
    0x6A09E667, 0xBB67AE85, 0x3C6EF372, 0xA54FF53A,
    0x510E527F, 0x9B05688C, 0x1F83D9AB, 0x5BE0CD19,
], dtype=np.uint32)

# Size in bits = number of boxes x 32.
print(f"block: {block.size} boxes = {block.size * 32} bits")
print(f"H0:    {H0.size} boxes = {H0.size * 32} bits")
assert block.dtype == np.uint32 and block.size * 32 == 512
assert H0.dtype == np.uint32 and H0.size * 32 == 256

# Print every box as 8 hex digits.
print("block =", " ".join(f"{int(w):08x}" for w in block))
print("H0    =", " ".join(f"{int(w):08x}" for w in H0))

block: 16 boxes = 512 bits
H0:    8 boxes = 256 bits
block = 61626380 00000000 00000000 00000000 00000000 00000000 00000000 00000000 00000000 00000000 00000000 00000000 00000000 00000000 00000000 00000018
H0    = 6a09e667 bb67ae85 3c6ef372 a54ff53a 510e527f 9b05688c 1f83d9ab 5be0cd19


### Input messages

SHA-256 takes a message of any length and works on its bits. In the standard, a message is a string of $\ell$ bits, where $0 \leq \ell < 2^{64}$ ([FIPS 180-4, Section 6.2, p.21](https://nvlpubs.nist.gov/nistpubs/FIPS/NIST.FIPS.180-4.pdf#page=26)). Computers store text as bytes, so we represent a message as a sequence of bytes.

#### Bytes

A byte is 8 bits, so it holds a whole number from $0$ to $2^8 - 1 = 255$. It is written as 2 hex digits (each digit is 4 bits). A message of $n$ bytes has length $\ell = 8n \text{ bits.}$

#### Text to bytes

Text has to be turned into bytes first using [UTF-8](https://developer.mozilla.org/en-US/docs/Glossary/UTF-8), where each ASCII character (like `a`) is exactly one byte. The same text always gives the same bytes, and the same bytes always give the same hash.

| Character | Byte (hex) | Byte (binary) |
|:---:|:---:|:---:|
| `a` | `61` | `01100001` |
| `b` | `62` | `01100010` |
| `c` | `63` | `01100011` |

So the text `"abc"` is the 3 bytes `61 62 63`, which is $\ell = 24$ bits.

#### In Python

- A Python [`bytes`](https://docs.python.org/3/library/stdtypes.html#bytes-objects) object is an immutable sequence of bytes. [`"abc".encode("utf-8")`](https://docs.python.org/3/library/stdtypes.html#str.encode) gives `b"abc"`.
- The `b` in front of the quotes is not part of the data. It is Python's way of saying "this is `bytes`, not text".
- A NumPy array with `dtype=np.uint8` is the same data as numbers $0$ to $255$, one per byte ([`numpy.frombuffer`](https://numpy.org/doc/stable/reference/generated/numpy.frombuffer.html) creates it without copying the bytes). This matches how we store words as `np.uint32` arrays.
- An empty message `b""` is valid - it has $\ell = 0$ bits and has a hash too.

In [4]:
import numpy as np

# Text becomes bytes using UTF-8. Printing it shows b'abc': the b means bytes.
message = "abc".encode("utf-8")

# The same bytes as a NumPy array: one uint8 number per byte.
# See: https://numpy.org/doc/stable/reference/generated/numpy.frombuffer.html
data = np.frombuffer(message, dtype=np.uint8)

print("bytes:  ", message)
print("numbers:", data)
print("hex:    ", " ".join(f"{b:02x}" for b in message))
print("binary: ", " ".join(f"{b:08b}" for b in message))

# Length in bits = number of bytes x 8.
length_bits = len(message) * 8
print("length: ", length_bits, "bits")

assert data.dtype == np.uint8
assert message.hex() == "616263"
assert length_bits == 24
# The empty message has length 0.
assert len(b"") * 8 == 0

bytes:   b'abc'
numbers: [97 98 99]
hex:     61 62 63
binary:  01100001 01100010 01100011
length:  24 bits


### 512-bit message block

SHA-256 does not read a whole message at once. It reads it in equal-sized pieces called blocks, one at a time ([FIPS 180-4, Section 5.2.1, p.14](https://nvlpubs.nist.gov/nistpubs/FIPS/NIST.FIPS.180-4.pdf#page=19)). Every block is 512 bits.

#### Block size

| Unit | Size | Units in one block |
|:---|:---:|:---:|
| bit | 1 bit | 512 |
| byte | 8 bits | $512 / 8 = 64$ |
| word | 32 bits | $512 / 32 = 16$ |

So one block is 64 bytes or 16 words. They are the same bits, just grouped differently.

#### Representation

We store one block as a NumPy array of 16 `np.uint32` values, where $M_j$ is `block[j]`.

```text
index:    0     1     2    ...   15
        +-----+-----+-----+-----+------+
block:  | M_0 | M_1 | M_2 | ... | M_15 |
        +-----+-----+-----+-----+------+
```


#### 64 bytes -> 16 words

A word is 4 bytes ($4 \times 8 = 32$). So we take the bytes four at a time, from left to right. The first 4 bytes make word 0, the next 4 bytes make word 1, and so on:

```text
bytes:   b0 b1 b2 b3 | b4 b5 b6 b7 | b8 b9 b10 b11 | ... | b60 b61 b62 b63
          \_________/   \_________/   \___________/           \____________/
words:       M0            M1             M2         ...          M15
```

In each word, the first byte is the biggest part, like the first digit in the number 123 (the hundreds). This is called big-endian. Example: the bytes `61 62 63 80` make the word `0x61626380`. The byte `61` goes first (biggest part) and `80` goes last (smallest part).

#### Example: `"abc"` block

The message `"abc"` is short, so it is padded up to 64 bytes. Bytes are counted from 0:

| Bytes | Value | What it is |
|:---:|:---|:---|
| 0 to 2 | `61 62 63` | the message `"abc"` |
| 3 | `80` | a `1` bit followed by seven `0` bits (padding) |
| 4 to 55 | `00 ... 00` | zeros (padding) |
| 56 to 63 | `00 00 00 00 00 00 00 18` | the message length: 24 bits, which is `0x18` |

Why is byte 3 equal to `80`? The padding rule is to add a single `1` bit straight after the message, then `0` bits. As one byte, that is `10000000` in binary, which is $2^7 = 128 =$ `0x80`. It marks where the message ends ([FIPS 180-4, Section 5.1.1, p.13](https://nvlpubs.nist.gov/nistpubs/FIPS/NIST.FIPS.180-4.pdf#page=18)). The [NIST SHA-256 example](https://csrc.nist.gov/CSRC/media/Projects/Cryptographic-Standards-and-Guidelines/documents/examples/SHA256.pdf) shows the same padded block for `"abc"`.

We take this finished block as given here.

In [5]:
import numpy as np

# 64 bytes for "abc": the message, then 0x80, then zeros, then the length 24 in the last 8 bytes.
block_bytes = b"abc" + b"\x80" + bytes(52) + (24).to_bytes(8, "big")
print(len(block_bytes), "bytes =", len(block_bytes) * 8, "bits")

# 16 empty boxes, then fill box j from bytes 4j to 4j+3 (first byte = biggest part).
# See: https://docs.python.org/3/library/stdtypes.html#int.from_bytes
block = np.zeros(16, dtype=np.uint32)
for j in range(16):
    four_bytes = block_bytes[4 * j : 4 * j + 4]
    block[j] = int.from_bytes(four_bytes, "big")

print("block =", " ".join(f"{int(w):08x}" for w in block))

assert len(block_bytes) == 64
assert block.dtype == np.uint32 and block.size * 32 == 512
# Box 0 is the first 4 bytes, box 15 is the length, and everything in between is zero.
assert block[0] == 0x61626380 and block[15] == 0x18
assert all(block[1:15] == 0)

64 bytes = 512 bits
block = 61626380 00000000 00000000 00000000 00000000 00000000 00000000 00000000 00000000 00000000 00000000 00000000 00000000 00000000 00000000 00000018


#### NumPy shortcut

The loop works, but NumPy can do the same job in one line. The trick is the `dtype` text `">u4"` (see [NumPy `dtype` reference](https://numpy.org/doc/stable/reference/arrays.dtypes.html) for the byte-order codes). Read it as three pieces:

| Piece | Meaning |
|:---:|:---|
| `>` | big-endian: the first byte is the biggest part (like the loop above) |
| `u` | unsigned: whole numbers, never negative |
| `4` | 4 bytes per number, which is 32 bits |

So `">u4"` means: "take the bytes four at a time, and read each group as one big-endian 32-bit whole number". That is exactly what the loop did. `.astype(np.uint32)` then converts the result to a normal `np.uint32` array. The numbers stay the same, only the way the computer stores them inside changes, so we can keep using one type.

In [ ]:
# One line instead of the loop.
# See: https://numpy.org/doc/stable/reference/arrays.dtypes.html
words = np.frombuffer(block_bytes, dtype=">u4").astype(np.uint32)

# It gives the same block as the loop.
assert np.array_equal(words, block)
print("same as the loop:", np.array_equal(words, block))

same as the loop: True


#### Multiple blocks

A long message needs several blocks, so we keep them in a 2-dimensional array. Each row is one block of 16 words. With $N$ blocks, the table has $N$ rows and 16 columns.

We first make one long list of words, then [`.reshape`](https://numpy.org/doc/stable/reference/generated/numpy.reshape.html) cuts it into rows:

```text
words:             [ w0  w1  ... w15  w16  w17  ... w31 ]
reshape(-1, 16):   block 0 = [ w0  w1  ... w15 ]
                   block 1 = [ w16 w17 ... w31 ]
```

`.reshape(-1, 16)` means "make rows of 16". The `-1` means "NumPy, you work out how many rows are needed". 32 words give 2 rows.

The example below is 128 bytes with the values $0, 1, 2, \dots, 127$, so the byte in position $k$ has the value $k$. That makes it easy to see where every byte ends up. Since $128 / 64 = 2$, we expect 2 blocks.

In [ ]:
# reshape on its own: 8 numbers become rows of 4 (-1 = work out the number of rows).
# See: https://numpy.org/doc/stable/reference/generated/numpy.reshape.html
print(np.arange(8).reshape(-1, 4))

# Not a real message: 128 bytes with the values 0, 1, 2, ... 127 (2 blocks of 64 bytes).
data = bytes(range(128))
words = np.frombuffer(data, dtype=">u4").astype(np.uint32)  # 32 words
blocks = words.reshape(-1, 16)                              # 2 rows of 16 words

print("shape:", blocks.shape, "= (number of blocks, words per block)")
for i, row in enumerate(blocks):
    print(f"block {i} =", " ".join(f"{int(w):08x}" for w in row))

assert blocks.shape == (2, 16)
assert blocks[0, 0] == 0x00010203 and blocks[1, 0] == 0x40414243

[[0 1 2 3]
 [4 5 6 7]]
shape: (2, 16) = (number of blocks, words per block)
block 0 = 00010203 04050607 08090a0b 0c0d0e0f 10111213 14151617 18191a1b 1c1d1e1f 20212223 24252627 28292a2b 2c2d2e2f 30313233 34353637 38393a3b 3c3d3e3f
block 1 = 40414243 44454647 48494a4b 4c4d4e4f 50515253 54555657 58595a5b 5c5d5e5f 60616263 64656667 68696a6b 6c6d6e6f 70717273 74757677 78797a7b 7c7d7e7f


#### Reading the result

- `shape: (2, 16)` means 2 blocks (rows) with 16 words (columns) each. This matches $128 / 64 = 2$.
- Block 0 starts with `00010203`. These are the first 4 bytes `00 01 02 03` joined into one word. The next word, `04050607`, is bytes 4 to 7, and so on.
- Block 0 ends with `3c3d3e3f`, which is bytes 60 to 63 (`0x3c` is 60).
- Block 1 starts at byte 64, which is `0x40` in hex. So its first word is `40414243`.

Every word is 4 bytes in a row, and every block is 16 words in a row.

### Final 256-bit hash value

### Big-endian integers

## Problem 2: Implement SHA-256 bitwise operations

## Problem 3: Generate SHA-256 constants

## Problem 4: Implement SHA-256 padding and block parsing

## Problem 5: Implement the SHA-256 compression function

## Problem 6: Complete and verify SHA-256